# Stage 06-D: Dataset Preparation

Builds a **self-contained HDF5 cache** for Stage 06-D end-to-end fine-tuning.
This notebook joins three data sources and writes `stage06d_cache/{split}.h5`.

## Data sources and join strategy

```
HuggingFace tranthaihoa/vifactcheck
  Statement (claim text)  ─────┐
  Context   (article text)     │  join by Url == metadata.source_url
  Evidence  (retrieved evid.)  │
  labels (0=Real,1=Fake,2=NEI) ─────────────────────────────────────┐
  Url ─────────────────────────┘                                     │
                                                                      │
Crawled JSON  news_data_vifactcheck_{split}_labeled.json             │
  id          (hex article ID)  ──────────────────────────────────── │
  metadata.source_url ──────────────────────────────────────────────┘
  media.images[].caption  ─────┐  group by article position (idx)
  media.images[].path     ─────┤
                               │
Pairs cache  pairs_{split}.json│
  article_idx  ───────────────┘  (position in crawled JSON)
  caption  (cleaned caption text)
  image_path  (full resolved path)

Stage 05a cache  stage05a_{split}.h5
  image_features [N, max_pairs=32, 64]  COOLANT image_aligned_clip
  pair_counts [N]
  labels [N]
```

## Output schema: `stage06d_cache/{split}.h5`

| Dataset          | Shape              | dtype   | Description                           |
| ---------------- | ------------------ | ------- | ------------------------------------- |
| `statements`     | [N]                | bytes   | Vietnamese claim text                 |
| `contexts`       | [N]                | bytes   | Full article context                  |
| `evidences`      | [N]                | bytes   | Retrieved evidence sentences          |
| `image_features` | [N, max_pairs, 64] | float32 | COOLANT image_aligned_clip            |
| `captions`       | [N, max_pairs]     | bytes   | Caption per image pair (empty=padded) |
| `pair_counts`    | [N]                | int32   | Real (unpadded) pairs per article     |
| `labels`         | [N]                | int64   | 0=Real 1=Fake 2=NEI                   |
| `article_ids`    | [N]                | bytes   | Hex ID from crawled JSON              |
| `urls`           | [N]                | bytes   | Source URL                            |

**Run this ONCE before `06d_finetune_fusion.ipynb`.**


In [1]:
import os, sys
from pathlib import Path


def _detect_platform():
    try:
        import google.colab

        return "colab", False
    except ImportError:
        pass
    if Path("/workspace").exists() and os.environ.get("VAST_CONTAINERLABEL"):
        return "vastai", False
    if Path("/workspace").exists():
        return "vastai", True
    if sys.platform == "win32":
        return "windows", False
    if sys.platform == "darwin":
        return "mac", False
    return None, True


PLATFORM, _ = _detect_platform()
if PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()
PROJECT_ROOT = (
    Path("/content/drive/MyDrive/Thesis_Final/fake-news-detection")
    if PLATFORM == "colab"
    else _nb_path.parents[1]
)
sys.path.insert(0, str(PROJECT_ROOT))
_env_map = {
    "colab": ".env.colab",
    "vastai": ".env.vastai",
    "windows": ".env.windows",
    "mac": ".env.mac",
}
_env_file = PROJECT_ROOT / _env_map.get(PLATFORM, ".env")
if not _env_file.exists():
    _env_file = PROJECT_ROOT / ".env"

from dotenv import load_dotenv

load_dotenv(_env_file, override=True)
from src.utils.env_utils import get_data_root

DATA_ROOT = get_data_root()
print(f"Platform: {PLATFORM}  DATA_ROOT: {DATA_ROOT}  exists={DATA_ROOT.exists()}")

Platform: mac  DATA_ROOT: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis  exists=True


In [2]:
# ── Configuration ─────────────────────────────────────────────────────────
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "pipeline" else Path.cwd()
try:
    from dotenv import load_dotenv

    load_dotenv(PROJECT_ROOT / ".env", override=False)
except ImportError:
    pass
DATA_ROOT = (
    Path(os.environ["DATA_ROOT"]) if os.environ.get("DATA_ROOT") else PROJECT_ROOT
)

CONFIG = {
    "paths": {
        "vifactcheck_json_dir": DATA_ROOT / "data" / "json",
        "pairs_cache_dir": DATA_ROOT / "processed_data" / "hdf5" / "pairs",
        "stage05a_cache_dir": DATA_ROOT / "training" / "stage05a_cache",
        "output_dir": DATA_ROOT / "training" / "stage06d_cache",
    },
    "data": {
        "hf_dataset": "tranthaihoa/vifactcheck",
        "splits": ["train", "dev", "test"],
        "hf_split_map": {},  # HF uses 'train'/'dev'/'test' directly
        "max_pairs": 32,  # must match stage05a cache
        "image_dim": 64,
        # Label mapping from HF labels field (0=Supported/Real, 1=Refuted/Fake, 2=NEI)
        "label_map": {0: 0, 1: 1, 2: 2},
    },
    # Set True to overwrite existing cache files
    "force_rebuild": False,
}

CONFIG["paths"]["output_dir"].mkdir(parents=True, exist_ok=True)
print(f'Output: {CONFIG["paths"]["output_dir"]}')
print(
    f'max_pairs={CONFIG["data"]["max_pairs"]}  image_dim={CONFIG["data"]["image_dim"]}'
)

Output: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/training/stage06d_cache
max_pairs=32  image_dim=64


In [3]:
# ── Dependencies ───────────────────────────────────────────────────────────
import json, gc
from collections import defaultdict
from datetime import datetime

import numpy as np
import h5py
from tqdm.auto import tqdm

print("Dependencies OK")

Dependencies OK


## Step 1: Load ViFactCheck Claims (Statement + Context + Evidence + Label)

Source: HuggingFace `tranthaihoa/vifactcheck`  
Join key: `Url` field → matched to `metadata.source_url` in crawled JSON


In [4]:
from datasets import load_dataset

hf_splits = {}
for split in CONFIG["data"]["splits"]:
    hf_split_name = CONFIG["data"]["hf_split_map"].get(split, split)
    ds = load_dataset(CONFIG["data"]["hf_dataset"], split=hf_split_name)
    # Index by URL (strip trailing slash for robust matching)
    by_url = {}
    for row in ds:
        url = (row.get("Url") or "").rstrip("/")
        if url:
            by_url[url] = {
                "statement": str(row.get("Statement", "")),
                "context": str(row.get("Context", "")),
                "evidence": str(row.get("Evidence", "")),
                "label": CONFIG["data"]["label_map"][int(row["labels"])],
                "url": url,
            }
    hf_splits[split] = by_url
    print(f"  [{split}] {len(ds)} HF records  indexed={len(by_url)} URLs")

print("✅ HF ViFactCheck loaded.")

  [train] 5062 HF records  indexed=1035 URLs
  [dev] 723 HF records  indexed=496 URLs
  [test] 1447 HF records  indexed=758 URLs
✅ HF ViFactCheck loaded.


## Step 2: Load Crawled JSON + Pairs Cache

Crawled JSON gives: `id` (hex article_id), `metadata.source_url` → join key to HF  
Pairs cache gives: `article_idx` (position in crawled JSON), `caption`, `image_path`

Groups captions by article_idx for later padding.


In [5]:
def load_crawled_and_pairs(split):
    """
    Returns:
        crawled: list of article dicts (raw JSON)
        pairs_by_idx: dict[int -> list[dict]]  article_idx -> sorted pairs
    """
    json_path = (
        CONFIG["paths"]["vifactcheck_json_dir"]
        / f"news_data_vifactcheck_{split}_labeled.json"
    )
    if not json_path.exists():
        raise FileNotFoundError(f"Crawled JSON not found: {json_path}")
    with open(json_path, encoding="utf-8") as f:
        crawled = json.load(f)

    pairs_path = CONFIG["paths"]["pairs_cache_dir"] / f"pairs_{split}.json"
    if not pairs_path.exists():
        raise FileNotFoundError(
            f"Pairs cache not found: {pairs_path}\n"
            "Run 02_preprocessing.ipynb first (Step 2: Extract Valid Matched Pairs)."
        )
    with open(pairs_path, encoding="utf-8") as f:
        pairs = json.load(f)

    # Group pairs by article index
    pairs_by_idx = defaultdict(list)
    for p in pairs:
        pairs_by_idx[int(p["article_idx"])].append(p)

    print(
        f"  [{split}] crawled={len(crawled)}  pairs={len(pairs)}  "
        f"articles_with_pairs={len(pairs_by_idx)}"
    )
    return crawled, pairs_by_idx


crawled_data = {}
pairs_data = {}
for split in CONFIG["data"]["splits"]:
    crawled_data[split], pairs_data[split] = load_crawled_and_pairs(split)
print("✅ Crawled JSON + pairs cache loaded.")

  [train] crawled=5062  pairs=6724  articles_with_pairs=3304
  [dev] crawled=723  pairs=862  articles_with_pairs=468
  [test] crawled=1447  pairs=2053  articles_with_pairs=967
✅ Crawled JSON + pairs cache loaded.


## Step 3: Load Stage 05a Image Features

Source: `stage05a_{split}.h5`  
Contains pre-extracted COOLANT `image_aligned_clip` features [N, max_pairs, 64].  
We align by position (article index in the crawled JSON array).


In [6]:
def load_stage05a(split):
    """
    Returns:
        img_feats:   np.array [N, max_pairs, 64]
        pair_counts: np.array [N]
        labels05a:   np.array [N]
        art_ids05a:  list[str]  article hex IDs (if stored) else positional
    """
    p = CONFIG["paths"]["stage05a_cache_dir"] / f"stage05a_{split}.h5"
    if not p.exists():
        raise FileNotFoundError(
            f"Stage 05a cache missing: {p}\n"
            "Run 05a_mil_attention_fusion.ipynb first."
        )
    with h5py.File(p, "r") as f:
        img_feats = f["image_features"][:].astype(np.float32)
        pair_counts = f["pair_counts"][:].astype(np.int32)
        labels05a = f["labels"][:].astype(np.int64)
        n = int(f.attrs["n_articles"])
        max_pairs = int(f.attrs["max_pairs"])
        if "article_ids" in f:
            art_ids = [
                s.decode() if isinstance(s, bytes) else str(s)
                for s in f["article_ids"][:]
            ]
        else:
            art_ids = [str(i) for i in range(n)]

    if max_pairs != CONFIG["data"]["max_pairs"]:
        raise ValueError(
            f'max_pairs mismatch: stage05a={max_pairs} vs CONFIG={CONFIG["data"]["max_pairs"]}'
        )
    print(f"  [{split}] n={n}  img_feats={img_feats.shape}  max_pairs={max_pairs}")
    return img_feats, pair_counts, labels05a, art_ids


stage05a = {}
for split in CONFIG["data"]["splits"]:
    stage05a[split] = load_stage05a(split)
print("✅ Stage 05a image features loaded.")

  [train] n=5062  img_feats=(5062, 32, 64)  max_pairs=32
  [dev] n=723  img_feats=(723, 32, 64)  max_pairs=32
  [test] n=1447  img_feats=(1447, 32, 64)  max_pairs=32
✅ Stage 05a image features loaded.


## Step 4: Join All Sources and Build Records

For each article (by position in crawled JSON):

1. Look up `metadata.source_url` → HF record (Statement, Context, Evidence, label)
2. Look up position → stage05a row (image_features, pair_counts)
3. Look up article_idx → pairs_by_idx (raw captions list)

Prints join statistics.


In [7]:
MAX_PAIRS = CONFIG["data"]["max_pairs"]


def build_records(split):
    crawled = crawled_data[split]
    pairs_by_idx = pairs_data[split]
    hf_by_url = hf_splits[split]
    img_feats, pair_counts, labels05a, art_ids = stage05a[split]

    if len(crawled) != len(img_feats):
        raise ValueError(
            f"[{split}] Crawled JSON has {len(crawled)} articles but "
            f"stage05a has {len(img_feats)} rows. Run stages in order."
        )

    records = []
    stats = {"hf_matched": 0, "hf_missing": 0, "has_images": 0, "no_images": 0}

    for idx, article in enumerate(tqdm(crawled, desc=f"[{split}] building")):
        # Join key: URL
        url = article.get("metadata", {}).get("source_url", "").rstrip("/")
        hf_rec = hf_by_url.get(url)
        if hf_rec is None:
            stats["hf_missing"] += 1
            # Fall back: empty strings, label from stage05a
            statement = ""
            context = ""
            evidence = ""
            label = int(labels05a[idx])
        else:
            stats["hf_matched"] += 1
            statement = hf_rec["statement"]
            context = hf_rec["context"]
            evidence = hf_rec["evidence"]
            label = hf_rec["label"]

        # Article hex ID from crawled JSON
        article_id = str(article.get("id", art_ids[idx]))

        # Captions: collect from pairs cache, pad to max_pairs
        article_pairs = pairs_by_idx.get(idx, [])
        captions_raw = [p.get("caption", "") for p in article_pairs[:MAX_PAIRS]]
        # Pad
        while len(captions_raw) < MAX_PAIRS:
            captions_raw.append("")

        if article_pairs:
            stats["has_images"] += 1
        else:
            stats["no_images"] += 1

        records.append(
            {
                "article_id": article_id,
                "url": url,
                "statement": statement,
                "context": context,
                "evidence": evidence,
                "label": label,
                "image_feat": img_feats[idx],  # [max_pairs, 64]
                "pair_count": int(pair_counts[idx]),
                "captions": captions_raw,  # list[str], len=max_pairs
            }
        )

    print(f"[{split}] {len(records)} records built")
    print(f'  HF matched={stats["hf_matched"]}  missing={stats["hf_missing"]}')
    print(f'  has_images={stats["has_images"]}  no_images={stats["no_images"]}')
    label_hist = np.bincount([r["label"] for r in records], minlength=3).tolist()
    print(f"  label_hist (Real/Fake/NEI): {label_hist}")
    return records


all_records = {}
for split in CONFIG["data"]["splits"]:
    all_records[split] = build_records(split)
print("✅ Join complete.")

[train] building:   0%|          | 0/5062 [00:00<?, ?it/s]

[train] 5062 records built
  HF matched=5062  missing=0
  has_images=3304  no_images=1758
  label_hist (Real/Fake/NEI): [1808, 1747, 1507]


[dev] building:   0%|          | 0/723 [00:00<?, ?it/s]

[dev] 723 records built
  HF matched=723  missing=0
  has_images=468  no_images=255
  label_hist (Real/Fake/NEI): [259, 242, 222]


[test] building:   0%|          | 0/1447 [00:00<?, ?it/s]

[test] 1447 records built
  HF matched=1447  missing=0
  has_images=967  no_images=480
  label_hist (Real/Fake/NEI): [501, 443, 503]
✅ Join complete.


## Step 5: Write HDF5 Cache

Writes `stage06d_cache/{split}.h5` with all aligned fields.
Text fields stored as variable-length UTF-8 bytes (`h5py.special_dtype(vlen=str)`).


In [8]:
def write_hdf5(split, records):
    out_path = CONFIG["paths"]["output_dir"] / f"{split}.h5"

    if out_path.exists() and not CONFIG["force_rebuild"]:
        print(
            f"  [{split}] {out_path.name} exists, skipping "
            f"(set force_rebuild=True to overwrite)"
        )
        return out_path

    N = len(records)
    MAX_P = CONFIG["data"]["max_pairs"]
    IMG_DIM = CONFIG["data"]["image_dim"]

    # Pre-allocate arrays
    img_arr = np.zeros((N, MAX_P, IMG_DIM), dtype=np.float32)
    cnt_arr = np.zeros(N, dtype=np.int32)
    lbl_arr = np.zeros(N, dtype=np.int64)

    str_fields = {
        k: [] for k in ("article_ids", "urls", "statements", "contexts", "evidences")
    }
    cap_list = []  # list of list[str], will be stored as object array

    for i, rec in enumerate(records):
        img_arr[i] = rec["image_feat"]
        cnt_arr[i] = rec["pair_count"]
        lbl_arr[i] = rec["label"]
        str_fields["article_ids"].append(rec["article_id"])
        str_fields["urls"].append(rec["url"])
        str_fields["statements"].append(rec["statement"])
        str_fields["contexts"].append(rec["context"])
        str_fields["evidences"].append(rec["evidence"])
        cap_list.append(rec["captions"])  # list of MAX_P strings

    vlen_str = h5py.special_dtype(vlen=str)

    with h5py.File(out_path, "w") as f:
        # Numeric arrays
        f.create_dataset(
            "image_features", data=img_arr, compression="gzip", compression_opts=4
        )
        f.create_dataset("pair_counts", data=cnt_arr)
        f.create_dataset("labels", data=lbl_arr)

        # String scalar arrays (1-D)
        for name, lst in str_fields.items():
            ds = f.create_dataset(name, shape=(N,), dtype=vlen_str)
            for j, s in enumerate(lst):
                ds[j] = s

        # Captions: 2-D variable-length string [N, max_pairs]
        cap_ds = f.create_dataset("captions", shape=(N, MAX_P), dtype=vlen_str)
        for i, row in enumerate(cap_list):
            for j, cap in enumerate(row):
                cap_ds[i, j] = cap

        # Metadata attributes
        f.attrs["n_articles"] = N
        f.attrs["max_pairs"] = MAX_P
        f.attrs["image_dim"] = IMG_DIM
        f.attrs["split"] = split
        f.attrs["hf_dataset"] = CONFIG["data"]["hf_dataset"]
        f.attrs["created_at"] = datetime.now().isoformat()
        f.attrs["label_map"] = "0=Real(Supported) 1=Fake(Refuted) 2=NEI"
        f.attrs["text_fields"] = "statement, context, evidence"
        f.attrs["image_source"] = "stage05a COOLANT image_aligned_clip"

    size_mb = out_path.stat().st_size / (1024**2)
    print(f"  [{split}] Written: {out_path.name}  ({size_mb:.1f} MB)  n={N}")
    return out_path


for split in CONFIG["data"]["splits"]:
    write_hdf5(split, all_records[split])

print("\n✅ stage06d_cache build complete.")

  [train] Written: train.h5  (31.4 MB)  n=5062
  [dev] Written: dev.h5  (4.4 MB)  n=723
  [test] Written: test.h5  (9.0 MB)  n=1447

✅ stage06d_cache build complete.


## Step 6: Validation — Inspect Cache Contents


In [9]:
import numpy as np

for split in CONFIG["data"]["splits"]:
    p = CONFIG["paths"]["output_dir"] / f"{split}.h5"
    if not p.exists():
        print(f"[{split}] MISSING")
        continue

    with h5py.File(p, "r") as f:
        n = f.attrs["n_articles"]
        max_p = f.attrs["max_pairs"]
        labels = f["labels"][:]
        pair_cnts = f["pair_counts"][:]
        img_shape = f["image_features"].shape
        # Read sample text fields
        stmt0 = f["statements"][0]
        ctx0 = f["contexts"][0]
        cap0 = f["captions"][0, 0]
        aid0 = f["article_ids"][0]

    label_hist = np.bincount(labels, minlength=3).tolist()
    print(f"[{split}]")
    print(f"  n={n}  max_pairs={max_p}  image_features={img_shape}")
    print(f"  label_hist (Real/Fake/NEI): {label_hist}")
    print(
        f"  pair_counts: min={pair_cnts.min()} max={pair_cnts.max()} "
        f"mean={pair_cnts.mean():.1f} zero={int((pair_cnts==0).sum())}"
    )
    print(f"  sample [0]:")
    print(f"    article_id : {aid0}")
    print(f"    statement  : {str(stmt0)[:100]}")
    print(f"    context    : {str(ctx0)[:100]}")
    print(f"    caption[0] : {str(cap0)[:80]}")
    print()

[train]
  n=5062  max_pairs=32  image_features=(5062, 32, 64)
  label_hist (Real/Fake/NEI): [1808, 1747, 1507]
  pair_counts: min=0 max=23 mean=1.3 zero=1758
  sample [0]:
    article_id : b'89d215119ab8eca5'
    statement  : b'Ph\xc6\xb0\xc6\xa1ng \xc3\xa1n \xc4\x91\xe1\xbb\x83 \xc4\x91\xe1\xba\xa3m b\xe1\xba\xa3o cung c\xe
    context    : b'(PLO)- Theo T\xe1\xbb\x95ng C\xc3\xb4ng ty C\xe1\xba\xa5p n\xc6\xb0\xe1\xbb\x9bc S\xc3\xa0i G\xc3\
    caption[0] : b''

[dev]
  n=723  max_pairs=32  image_features=(723, 32, 64)
  label_hist (Real/Fake/NEI): [259, 242, 222]
  pair_counts: min=0 max=15 mean=1.2 zero=255
  sample [0]:
    article_id : b'548ea82e0a698a1f'
    statement  : b'C\xc6\xa1 quan C\xe1\xba\xa3nh s\xc3\xa1t \xc4\x91i\xe1\xbb\x81u tra C\xc3\xb4ng an TP HCM \xc4\x9
    context    : b'(NL\xc4\x90O)- Sau khi mua kho\xe1\xba\xa3n n\xe1\xbb\xa3 t\xe1\xbb\xab C\xc3\xb4ng ty Mirae Asset
    caption[0] : b''

[test]
  n=1447  max_pairs=32  image_features=(1447, 32, 64)
  label_hist 

## Step 7: Usage in 06d Training Notebook

Replace the `load_vifactcheck_split` + `load_image_lookup` + `align_split` steps in
`06d_finetune_fusion.ipynb` with this single loader:


In [10]:
# ── Example: loading stage06d_cache in the training notebook ─────────────


def load_stage06d_split(split, cache_dir):
    """
    Load a stage06d cache split.

    Returns list of dicts:
      {
        'article_id': str,
        'statement' : str,   # ViFactCheck claim
        'context'   : str,   # Full article context
        'evidence'  : str,   # Retrieved evidence sentences
        'image'     : np.array [max_pairs, 64],
        'captions'  : list[str]  len=max_pairs,
        'pair_count': int,
        'label'     : int,
      }
    """
    p = Path(cache_dir) / f"{split}.h5"
    if not p.exists():
        raise FileNotFoundError(
            f"stage06d cache not found: {p}. Run 06d_prepare_dataset.ipynb first."
        )

    records = []
    with h5py.File(p, "r") as f:
        N = int(f.attrs["n_articles"])
        max_p = int(f.attrs["max_pairs"])
        imgs = f["image_features"][:]
        cnts = f["pair_counts"][:]
        lbls = f["labels"][:]
        stmts = f["statements"][:]
        ctxs = f["contexts"][:]
        evids = f["evidences"][:]
        aids = f["article_ids"][:]
        caps = f["captions"][:]  # [N, max_pairs] object array

    for i in range(N):
        records.append(
            {
                "article_id": str(aids[i]),
                "statement": str(stmts[i]),
                "context": str(ctxs[i]),
                "evidence": str(evids[i]),
                "image": imgs[i],
                "captions": [str(c) for c in caps[i]],
                "pair_count": int(cnts[i]),
                "label": int(lbls[i]),
            }
        )
    return records


# ── Test the loader ────────────────────────────────────────────────────────
cache_dir = CONFIG["paths"]["output_dir"]
test_recs = load_stage06d_split("train", cache_dir)
r = test_recs[0]
print(f"Loaded {len(test_recs)} records")
print(f'article_id  : {r["article_id"]}')
print(f'statement   : {r["statement"][:100]}')
print(f'context     : {r["context"][:100]}')
print(f'evidence    : {r["evidence"][:100]}')
print(f'image shape : {r["image"].shape}')
print(f'pair_count  : {r["pair_count"]}')
print(f'captions[0] : {r["captions"][0][:80]}')
print(f'label       : {r["label"]}  (0=Real 1=Fake 2=NEI)')
print()
print("Paste load_stage06d_split() into 06d_finetune_fusion.ipynb")
print("Replace Steps 4, 5, 6 (text + image loading + alignment) with this single call.")

Loaded 5062 records
article_id  : b'89d215119ab8eca5'
statement   : b'Ph\xc6\xb0\xc6\xa1ng \xc3\xa1n \xc4\x91\xe1\xbb\x83 \xc4\x91\xe1\xba\xa3m b\xe1\xba\xa3o cung c\xe
context     : b'(PLO)- Theo T\xe1\xbb\x95ng C\xc3\xb4ng ty C\xe1\xba\xa5p n\xc6\xb0\xe1\xbb\x9bc S\xc3\xa0i G\xc3\
evidence    : b'vi\xe1\xbb\x87c c\xc3\xbap n\xc6\xb0\xe1\xbb\x9bc l\xc3\xa0 \xc4\x91\xe1\xbb\x83 th\xe1\xbb\xb1c h
image shape : (32, 64)
pair_count  : 0
captions[0] : b''
label       : 0  (0=Real 1=Fake 2=NEI)

Paste load_stage06d_split() into 06d_finetune_fusion.ipynb
Replace Steps 4, 5, 6 (text + image loading + alignment) with this single call.


## Step 8: Dataset Flow Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

SPLITS = CONFIG["data"]["splits"]
LABEL_NAMES = ["Real", "Fake", "NEI"]
LABEL_COLORS = ["#4caf50", "#f44336", "#ff9800"]

# ── Collect per-split stats from all_records ──────────────────────────────
split_sizes = []
label_hists = []
has_img_counts = []
no_img_counts = []
pair_count_means = []
pair_count_all = {s: [] for s in SPLITS}

for split in SPLITS:
    recs = all_records[split]
    split_sizes.append(len(recs))
    labels = [r["label"] for r in recs]
    label_hists.append(np.bincount(labels, minlength=3).tolist())
    pc = [r["pair_count"] for r in recs]
    has_img_counts.append(sum(1 for x in pc if x > 0))
    no_img_counts.append(sum(1 for x in pc if x == 0))
    pair_count_means.append(np.mean(pc))
    pair_count_all[split] = pc

fig = plt.figure(figsize=(18, 12))
fig.suptitle("Stage 06-D Dataset Flow Overview", fontsize=16, fontweight="bold", y=0.98)

# ── 1. Split sizes ────────────────────────────────────────────────────────
ax1 = fig.add_subplot(2, 3, 1)
bars = ax1.bar(SPLITS, split_sizes, color=["#5c85d6", "#5ca85c", "#d6855c"], width=0.5)
for bar, v in zip(bars, split_sizes):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
             str(v), ha="center", va="bottom", fontsize=10, fontweight="bold")
ax1.set_title("Articles per Split")
ax1.set_ylabel("Count")
ax1.set_ylim(0, max(split_sizes) * 1.15)

# ── 2. Label distribution per split (grouped bars) ────────────────────────
ax2 = fig.add_subplot(2, 3, 2)
x = np.arange(len(SPLITS))
width = 0.22
for i, (name, color) in enumerate(zip(LABEL_NAMES, LABEL_COLORS)):
    vals = [label_hists[j][i] for j in range(len(SPLITS))]
    ax2.bar(x + (i - 1) * width, vals, width, label=name, color=color)
ax2.set_title("Label Distribution per Split")
ax2.set_ylabel("Count")
ax2.set_xticks(x)
ax2.set_xticklabels(SPLITS)
ax2.legend()

# ── 3. Image coverage (has-image vs no-image) ────────────────────────────
ax3 = fig.add_subplot(2, 3, 3)
x = np.arange(len(SPLITS))
ax3.bar(x, has_img_counts, label="Has images", color="#42a5f5")
ax3.bar(x, no_img_counts, bottom=has_img_counts, label="No images", color="#bdbdbd")
for i, (h, n) in enumerate(zip(has_img_counts, no_img_counts)):
    pct = 100 * h / (h + n)
    ax3.text(i, h / 2, f"{pct:.0f}%\nwith img", ha="center", va="center",
             color="white", fontsize=9, fontweight="bold")
ax3.set_title("Image Coverage per Split")
ax3.set_ylabel("Articles")
ax3.set_xticks(x)
ax3.set_xticklabels(SPLITS)
ax3.legend()

# ── 4. pair_count distribution (histogram, train) ─────────────────────────
ax4 = fig.add_subplot(2, 3, 4)
pc_train = [p for p in pair_count_all["train"] if p > 0]
ax4.hist(pc_train, bins=range(1, max(pc_train) + 2), color="#7e57c2", edgecolor="white", align="left")
ax4.set_title("Train: pair_count Distribution\n(articles with ≥1 image pair)")
ax4.set_xlabel("pair_count")
ax4.set_ylabel("# Articles")

# ── 5. Label distribution as stacked % (normalized) ───────────────────────
ax5 = fig.add_subplot(2, 3, 5)
label_pcts = np.array(label_hists, dtype=float)
label_pcts = label_pcts / label_pcts.sum(axis=1, keepdims=True) * 100
bottoms = np.zeros(len(SPLITS))
for i, (name, color) in enumerate(zip(LABEL_NAMES, LABEL_COLORS)):
    ax5.bar(SPLITS, label_pcts[:, i], bottom=bottoms, label=name, color=color)
    for j, (pct, bot) in enumerate(zip(label_pcts[:, i], bottoms)):
        ax5.text(j, bot + pct / 2, f"{pct:.0f}%", ha="center", va="center",
                 color="white", fontsize=9, fontweight="bold")
    bottoms += label_pcts[:, i]
ax5.set_title("Label Mix (%) per Split")
ax5.set_ylabel("% of Articles")
ax5.set_ylim(0, 105)
ax5.legend(loc="upper right")

# ── 6. Pipeline flow diagram ───────────────────────────────────────────────
ax6 = fig.add_subplot(2, 3, 6)
ax6.set_xlim(0, 10)
ax6.set_ylim(0, 10)
ax6.axis("off")
ax6.set_title("Pipeline Join Flow", fontsize=11)

boxes = [
    (5, 9.0, "HuggingFace\nvifactcheck\n(Statement / Context\n/ Evidence / Label)", "#4fc3f7"),
    (2, 6.5, "Crawled JSON\n(article_id, source_url)", "#81c784"),
    (8, 6.5, "pairs_{split}.json\n(article_idx, caption\nimage_path)", "#ffb74d"),
    (5, 4.0, "Join by URL + article_idx\n→ all_records[split]", "#ce93d8"),
    (2, 2.0, "stage05a_{split}.h5\n(image_features [N,32,64]\npair_counts, labels)", "#f48fb1"),
    (5, 0.5, "stage06d_cache/{split}.h5\nFinal HDF5 output", "#a5d6a7"),
]
for (x, y, txt, color) in boxes:
    ax6.add_patch(mpatches.FancyBboxPatch(
        (x - 1.7, y - 0.55), 3.4, 1.1,
        boxstyle="round,pad=0.1", facecolor=color, edgecolor="#555", linewidth=1.2, alpha=0.9))
    ax6.text(x, y, txt, ha="center", va="center", fontsize=7.5, fontweight="bold")

# Arrows
arrows = [
    (5, 8.45, 5, 4.55),      # HF → Join
    (2, 5.95, 3.5, 4.55),    # Crawled → Join
    (8, 5.95, 6.5, 4.55),    # Pairs → Join
    (2, 1.45, 3.5, 0.55),    # stage05a → output (via join)
    (5, 3.45, 5, 1.05),      # Join → output
]
for (x1, y1, x2, y2) in arrows:
    ax6.annotate("", xy=(x2, y2), xytext=(x1, y1),
                 arrowprops=dict(arrowstyle="->", color="#333", lw=1.5))

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("dataset_flow.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved → dataset_flow.png")